## 0. Подготовка

In [1]:
import os

import pandas as pd

from solution_utils import (
    reaction_metrics, init_reaction_models, partition_scores, predict,
    read_dialogs, save_predictions, init_topic_models, read_labels, compute_metrics, show_table
)
import solution_utils as su

In [2]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

диалогов: 10000


### Примеры работы функции чистки

In [3]:
text = """Привет! Смотри https://example.com/page?id=42&x=1 и www.test.org — там всё есть.
Вот формула: \theta + \nu = \rho, и ещё \frac{1}{2} \cdot \sqrt{x}.
Код: 
```python
def foo(x): 
    return x ** 2 + 1
Текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст текст текст
текст текст текст текст текст текст текст текст текст текст текст."""

print(su.clean_text(text))

Привет! Смотри и — там всё есть. Вот формула: heta + Код: zzcode zzmath zzurl


<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_185758/570074093.py:2: SyntaxWarning: invalid escape sequence '\c'
  Вот формула: \theta + \nu = \rho, и ещё \frac{1}{2} \cdot \sqrt{x}.


### Пример работы стемминга

In [4]:
from artifacts import load_lid_model, predict_language
words = ["cats", "кошки", "chattes", "Katzen", "яблочный", 'trees']
model, vectorizer = load_lid_model()
stemmer = su.Stemmer()

results = []
for w in words:
    language = predict_language(w, model, vectorizer)
    stem = stemmer.stem(w)
    results.append({"слово": w, "язык": language, "стем": stem})

df = pd.DataFrame(results)
df

/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,слово,язык,стем
0,cats,english,cat
1,кошки,russian,кошки
2,chattes,french,chattes
3,Katzen,german,Katzen
4,яблочный,russian,яблочный
5,trees,english,tree


## Загрузка и обработка данных

In [5]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

dialogs = su.read_dialogs(DATA_PATH)

frame_all = su.to_frame(dialogs)
print("диалогов:", len(frame_all), " размечено по реакции:", frame_all.reaction.notna().sum())
df = frame_all[frame_all.reaction.notna()].reset_index(drop=True)
df[["dialog_id", "last_raw", "n_user", "reaction"]].head()

Обработано 1000 диалогов...
Обработано 2000 диалогов...
Обработано 3000 диалогов...
Обработано 4000 диалогов...
Обработано 5000 диалогов...
Обработано 6000 диалогов...
Обработано 7000 диалогов...
Обработано 8000 диалогов...
Обработано 9000 диалогов...
Обработано 10000 диалогов...
диалогов: 10000  размечено по реакции: 3000


,dialog_id,last_raw,n_user,reaction
0,d094012177e,"any spec symbols used in ""paradoxical arithmet...",7,positive
1,d306fbe61fe,Give me GUI for this portion,8,neutral
2,dd6905952c7,1.Alex was at the supermarket when a fan bumpe...,5,neutral
3,d767a675896,recalculate everything considering the porport...,6,negative
4,d7a6280ce8e,how many masses we have on spectrometer,8,negative


In [6]:
# связь реакции с простыми числовыми признаками
display(df.groupby("reaction")[["n_user", "len_last", "len_asst"]].median())
su.show_table(pd.crosstab(df.n_user, df.reaction, normalize="index").round(3))

,n_user,len_last,len_asst
reaction,,,
negative,6.0,78.0,1350.333333
neutral,6.0,85.0,1370.190476
positive,7.0,63.0,1098.625000


reaction,negative,neutral,positive
n_user,,,
5,0.280,0.625,0.095
6,0.320,0.598,0.082
7,0.312,0.572,0.116
8,0.344,0.554,0.102
9,0.309,0.561,0.130
10,0.305,0.577,0.118


## Задача 1

### Разбиение

### Baseline

In [7]:
reaction_models = init_reaction_models()
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics_tb = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict(model, df)
    if reaction_labels:
        reaction_metrics_tb[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels, reaction_metrics)

show_table(pd.DataFrame(reaction_metrics_tb).T.round(3))

,macro_f1,macro_f1_bal,accuracy,balanced_acc,P_negative,R_negative,F1_negative,P_neutral,R_neutral,F1_neutral,P_positive,R_positive,F1_positive
dummy_most_frequent,0.247,0.167,0.589,0.333,0.000,0.000,0.000,0.589,1.000,0.742,0.000,0.000,0.000
linear_svm,0.905,0.889,0.915,0.887,0.902,0.867,0.884,0.915,0.953,0.933,0.962,0.842,0.898
linear_svm_my,0.933,0.917,0.945,0.916,0.943,0.921,0.932,0.942,0.973,0.957,0.970,0.855,0.909
logreg,0.910,0.901,0.922,0.900,0.897,0.900,0.899,0.933,0.945,0.939,0.932,0.855,0.892
logreg_my,0.918,0.912,0.930,0.912,0.901,0.916,0.908,0.945,0.947,0.946,0.933,0.871,0.901
nb,0.820,0.803,0.829,0.800,0.728,0.808,0.766,0.872,0.856,0.864,0.953,0.736,0.831
nb_my,0.896,0.886,0.904,0.885,0.844,0.907,0.874,0.930,0.916,0.923,0.962,0.832,0.892


## Задача 2

In [8]:
topic_models = init_topic_models()
topic_labels = read_labels(LABELS_PATH, "topic") if LABELS_PATH else {}

topic_predictions, topic_scores = {}, {}
for name, model in topic_models.items():
    topic_predictions[name] = predict(model, df)
    # print(topic_predictions[name])
    if topic_labels:
        topic_scores[name] = compute_metrics(dialogs, topic_predictions[name], topic_labels, partition_scores)
        print(topic_scores[name])

show_table(pd.DataFrame(topic_scores).T.round(3))

In [9]:
save_predictions(PREDICTIONS_PATH, dialogs, {"reaction": reaction_predictions, "topic": topic_predictions})
print("строк в файле ответов:", len(dialogs) * (len(reaction_predictions) + len(topic_predictions)))

строк в файле ответов: 120000
